# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

p = Path('data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(p)
df['is_declining_label'] = df['trend_direction'].fillna('').str.lower() == 'down'
df['stale'] = (df['days_since_last_update'] >= 180).astype(int)
df['visible'] = (df['impressions_90d'] >= 500).astype(int)
df['baseline_score'] = df['stale'] * df['visible'] * df['impressions_90d']
# attach reason codes
def reason(row):
    codes = []
    if row['stale'] and row['visible']:
        codes.append('stale_visible')
    elif row['stale']:
        codes.append('stale_only')
    elif row['visible']:
        codes.append('visible_only')
    return '|'.join(codes) if codes else 'none'
df['reason_code'] = df.apply(reason, axis=1)
print('baseline score computed, sample:')
print(df[['baseline_score','reason_code']].head())


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
print('Intended use: prioritize manual review for likely-declining content; do not auto-publish.')


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
checks = [
    'Verify impression definitions and time windows',
    'Check clients with low history depth',
    'Confirm no private client names in exports'
]
for c in checks:
    print('-', c)


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
print('Monitor precision@K drift, top-K reviewer rejection rate, and per-client data availability.')


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()
K = 50
if 'baseline_score' not in df.columns:
    print('baseline_score missing — run the first cell')
else:
    base_rate = df['is_declining_label'].mean()
    p_at_k = precision_at_k(df['baseline_score'], df['is_declining_label'], K)
    print(f'base rate: {base_rate:.3f}, precision@{K}: {p_at_k:.3f}')
    out = Path('work/outputs')
    out.mkdir(parents=True, exist_ok=True)
    topk = df.sort_values('baseline_score', ascending=False).head(200).copy()
    topk.to_csv(out / 'baseline_refresh_queue.csv', index=False)
    print('Exported top picks to', out / 'baseline_refresh_queue.csv')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.